In [1]:
!pip install numerapi pandas pyarrow lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.3 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [lightgbm]
   -------------------- ------------------- 1/2 [numerapi]
   ---------------------------------------- 2/2 [numerapi]



In [2]:
from numerapi import NumerAPI

napi = NumerAPI()
napi.download_dataset("v5.2/train.parquet", "train.parquet")
napi.download_dataset("v5.2/validation.parquet", "validation.parquet")
napi.download_dataset("v5.2/features.json", "features.json")

2026-07-01 14:03:28,600 INFO numerapi.utils: starting download
train.parquet: 100%|██████████████████████████████████████████████████████████████| 2.60G/2.60G [19:43<00:00, 2.19MB/s]
2026-07-01 14:23:13,495 INFO numerapi.utils: starting download
IOPub message rate exceeded.████████████████████████████████████████████████▍     | 3.90G/4.31G [28:15<02:57, 2.35MB/s]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

validation.parquet: 100%|█████████████████████████████████████████████████████████| 4.31G/4.31G [31:13<00:00, 2.30MB/s]
2026-07-01 14:54:28,308 INFO numerapi.utils: starting download
features.json: 100%|████████████████████████████████████████████████████████████████| 323k/323k [00:00<00:00, 1.12MB/s]


'features.json'

In [2]:
import pandas as pd

train = pd.read_parquet("train.parquet")
print(train.shape)
print(train.head())

(2746268, 2791)
                   era data_type  feature_shaded_hallucinatory_dactylology  \
id                                                                           
n0007b5abb0c3a25  0001     train                                         3   
n003bba8a98662e4  0001     train                                         4   
n003bee128c2fcfc  0001     train                                         2   
n0048ac83aff7194  0001     train                                         2   
n0055a2401ba6480  0001     train                                         4   

                  feature_itinerant_hexahedral_photoengraver  \
id                                                             
n0007b5abb0c3a25                                           4   
n003bba8a98662e4                                           2   
n003bee128c2fcfc                                           4   
n0048ac83aff7194                                           1   
n0055a2401ba6480                                     

In [4]:
target_cols = [c for c in train.columns if c.startswith("target_")]
print(target_cols)

['target_agnes_20', 'target_agnes_60', 'target_alpha_20', 'target_alpha_60', 'target_bravo_20', 'target_bravo_60', 'target_caroline_20', 'target_caroline_60', 'target_charlie_20', 'target_charlie_60', 'target_claudia_20', 'target_claudia_60', 'target_cyrusd_20', 'target_cyrusd_60', 'target_delta_20', 'target_delta_60', 'target_echo_20', 'target_echo_60', 'target_ender_20', 'target_ender_60', 'target_jasper_20', 'target_jasper_60', 'target_jeremy_20', 'target_jeremy_60', 'target_ralph_20', 'target_ralph_60', 'target_rowan_20', 'target_rowan_60', 'target_sam_20', 'target_sam_60', 'target_teager2b_20', 'target_teager2b_60', 'target_tyler_20', 'target_tyler_60', 'target_victor_20', 'target_victor_60', 'target_waldo_20', 'target_waldo_60', 'target_xerxes_20', 'target_xerxes_60']


In [6]:
import pandas as pd
import lightgbm as lgb
import json
import numpy as np

# 피처/타겟 분리
feature_cols = [c for c in train.columns if c.startswith("feature_")]
target_col = "target_cyrusd_20"  # 메인 타겟

# 메모리 절약: era 일부만 샘플링
eras = train["era"].unique()
sampled_eras = eras[:200]  # 전체 중 앞 200개 era만 사용
sample = train[train["era"].isin(sampled_eras)]

X = sample[feature_cols]
y = sample[target_col]

# LightGBM으로 feature importance 계산
model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# 중요도 정렬
importance = pd.Series(model.feature_importances_, index=feature_cols)
importance = importance.sort_values(ascending=False)

# 상위 50% 피처만 선택
threshold = importance.median()
selected_features = importance[importance >= threshold].index.tolist()

print(f"전체 피처 수: {len(feature_cols)}")
print(f"선택된 피처 수: {len(selected_features)}")
print(importance.head(20))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 10.951736 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12015
[LightGBM] [Info] Number of data points in the train set: 833533, number of used features: 2403
[LightGBM] [Info] Start training from score 0.500059
전체 피처 수: 2748
선택된 피처 수: 1517
feature_saharan_parietal_catch                   12
feature_scenic_unartistic_tache                  10
feature_delusory_fake_lower                      10
feature_probative_jolliest_february              10
feature_overstrong_burrier_guevara                9
feature_supplicatory_ski_broach                   9
feature_threadbare_reviving_franco                9
feature_droll_clumsy_gleaner                      9
feature_inartistic_necrological_showgirl          8
feature_catechetical_paragogical_accouterment     8
feature_roilier_slippered_patti                   7
feature_micrometrical_perspectival_luffa          7


In [7]:
# 선택된 피처 목록 저장
import json

with open("selected_features.json", "w") as f:
    json.dump(selected_features, f)

print(f"저장 완료! 선택된 피처 수: {len(selected_features)}")

저장 완료! 선택된 피처 수: 1517
